# Rising Waters: Flood Prediction - Full ML Pipeline
This notebook walks through dataset loading, EDA, preprocessing, model training/comparison, and saving the final model — matching the project specification step by step.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

df = pd.read_csv("flood_dataset.csv")
df.head()

## 1. Descriptive Statistics & Missing Values

In [ ]:
print(df.describe())
print("\nMissing values:\n", df.isnull().sum())

## 2. Univariate Analysis - Distribution Plots

In [ ]:
numeric_cols = ["annual_rainfall", "seasonal_rainfall", "cloud_visibility",
                "humidity", "temperature", "river_level"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

## 3. Box Plots - Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_cols + ["flood_occurred"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## 5. Data Preprocessing

In [ ]:
# Handle missing values
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# Outlier treatment (IQR capping)
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

X = df[numeric_cols]
y = df["flood_occurred"]

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Model Building & Comparison

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              eval_metric="logloss", random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    results[name] = (acc, model)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc*100:.2f}%")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

## 7. Save Best Model

In [ ]:
import joblib

best_name = max(results, key=lambda k: results[k][0])
best_acc, best_model = results[best_name]
print(f"Best model: {best_name} ({best_acc*100:.2f}%)")

joblib.dump(best_model, "floods.save")
joblib.dump(scaler, "scaler.save")
print("Saved floods.save and scaler.save")